# Annotation d'images et accord inter-annotateurs

Dans ce TP, vous allez :

1. écrire un guide d'annotation pour classer des photos de paysages ;
2. annoter 10 photos en suivant le guide d'une autre personne ;
3. mesurer l'accord inter-annotateurs, sur l'ensemble des photos puis photo par photo ;
4. réviser le guide là où il a divisé, puis annoter à nouveau ;
5. estimer la performance humaine de référence (HLP), face à un annotateur trivial.

Le TP se fait en groupe : le formateur rassemble les annotations de chacun. Seul, suivez-le avec les annotations d'exemple fournies, celles de cinq annotateurs fictifs, auxquelles s'ajoutent les vôtres.

## Récupération des données

In [ ]:
import pathlib
import urllib.request

!rm -rf sample_data

url = "https://github.com/shuuchuu/datasets/raw/refs/heads/main/landscape"


def telecharge(source, cible):
  """Télécharge un fichier du jeu des paysages, s'il n'est pas déjà là."""
  cible = pathlib.Path(cible)
  if not cible.exists():
    cible.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve(f"{url}/{source}", cible)


# Les 10 images à annoter, tirées des images sans étiquette du jeu
ids = [10034, 10159, 10244, 1060, 1072, 11, 11105, 1223, 12326, 12971]
for i, numero in enumerate(ids):
  telecharge(f"seg_pred/no_supervision/{numero}.jpg", f"a-annoter/{i}.jpg")

# Six exemples par classe, tirés du jeu d'entraînement
exemples = {
  "buildings": [4715, 6187, 11378, 18419, 9112, 8342],
  "forest": [5207, 1989, 7686, 3686, 17051, 8537],
  "glacier": [14073, 18597, 14079, 12794, 17566, 8191],
  "mountain": [9644, 14412, 19237, 12829, 122, 19857],
  "sea": [7078, 1345, 3731, 604, 2508, 17013],
  "street": [9049, 662, 5593, 8103, 17732, 11782],
}
for nom, numeros in exemples.items():
  for numero in numeros:
    telecharge(f"seg_train/{nom}/{numero}.jpg", f"exemples/{nom}/{numero}.jpg")

## Écriture d'un guide d'annotation

À partir des exemples de chaque classe, que la cellule suivante affiche, écrivez un guide d'annotation : une règle par classe, qui permette à quelqu'un qui n'a pas vu ces exemples de choisir, pour chaque photo, une des 6 classes : `buildings` (bâtiments), `forest` (forêt), `glacier`, `mountain` (montagne), `sea` (mer) et `street` (rue).

Écrivez-le dans la cellule de texte qui suit les images. En groupe, échangez-le ensuite avec celui d'une autre personne, comme le formateur l'indique.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image

classes = ["buildings", "forest", "glacier", "mountain", "sea", "street"]

fig, axes = plt.subplots(len(classes), 6, figsize=(12, 2.2 * len(classes)))
for axes_ligne, nom in zip(axes, classes):
  for ax, numero in zip(axes_ligne, exemples[nom]):
    ax.imshow(Image.open(f"exemples/{nom}/{numero}.jpg"))
    ax.axis("off")
  axes_ligne[0].set_title(nom, loc="left", fontweight="bold")
plt.tight_layout()

*Votre guide d'annotation*

## Annotation

Annotez les 10 photos du dossier `a-annoter` en suivant le guide d'une autre personne (seul : le vôtre), aussi fidèlement que possible, même quand vous auriez choisi autrement, et sans en discuter avec les autres.

Remplissez la liste `etiquettes` avec la classe que vous attribuez à chaque photo, dans l'ordre de 0 à 9 :

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for ax, i in zip(axes.flat, range(len(ids))):
  ax.imshow(Image.open(f"a-annoter/{i}.jpg"))
  ax.set_title(f"Image {i}")
  ax.axis("off")
plt.tight_layout()

In [ ]:
etiquettes = []  # À compléter, par exemple ["glacier", "sea", ...]

## Mise en commun des annotations

En groupe, donnez vos annotations au formateur, sur le support qu'il indique (un tableur partagé, par exemple : une ligne par participant, son nom puis les 10 classes). Collez ensuite les lignes de tout le groupe, les vôtres comprises, à la place des annotations d'exemple de la deuxième cellule ci-dessous, et passez `SEUL` à `False`. Les noms de classes en français (`bâtiments`, `forêt`, `montagne`, `mer`, `rue`) sont acceptés.

Seul, gardez les annotations d'exemple : vos annotations (`etiquettes`) s'y ajoutent, sous le nom `Moi`.

La première cellule définit les outils qui lisent les annotations et affichent les votes : il n'est pas nécessaire de lire son code.

In [ ]:
import re

import numpy


# Les noms de classes en français sont acceptés aussi
alias = {
  "bâtiments": "buildings",
  "bâtiment": "buildings",
  "batiments": "buildings",
  "batiment": "buildings",
  "forêt": "forest",
  "foret": "forest",
  "montagne": "mountain",
  "mer": "sea",
  "rue": "street",
  "route": "street",
}


def verifie(nom, etiquettes):
  """Vérifie qu'il y a une classe connue par image."""
  etiquettes = [
    alias.get(etiquette.lower(), etiquette.lower()) for etiquette in etiquettes
  ]
  inconnues = sorted(set(etiquettes) - set(classes))
  if inconnues or len(etiquettes) != len(ids):
    message = f"{nom} : {len(etiquettes)} classe(s), dont inconnue(s) {inconnues}, au lieu d'une par image parmi {classes}"
    raise ValueError(message)
  return etiquettes


def lit_annotations(texte):
  """Lit une ligne par annotateur : son nom, puis une classe par image."""
  annotations = {}
  for ligne in texte.strip().splitlines():
    nom, *etiquettes = re.split(r"[\s,;]+", ligne.strip())
    annotations[nom] = verifie(nom, etiquettes)
  return annotations


def affiche_votes(tableau, accord):
  """Affiche les images de la moins à la plus consensuelle, avec leurs votes."""
  fig, axes = plt.subplots(2, 5, figsize=(15, 7.2))
  for ax, i in zip(axes.flat, numpy.argsort(accord, kind="stable")):
    effectifs = sorted(zip(tableau[i], classes), reverse=True)
    detail = ", ".join(f"{nom} {effectif}" for effectif, nom in effectifs if effectif)
    ax.imshow(Image.open(f"a-annoter/{i}.jpg"))
    ax.set_title(
      f"Image {i} · accord {accord[i]:.0%}\n{detail}",
      color="black" if accord[i] == 1 else "tab:red",
    )
    ax.axis("off")
  plt.tight_layout()
  plt.show()

In [ ]:
SEUL = True  # En groupe : False, et les lignes du groupe à la place des exemples

# Une ligne par annotateur : son nom (en un mot), puis les 10 classes, séparés par
# des espaces, des virgules ou des tabulations (copier-coller depuis un tableur)
annotations = lit_annotations("""
Alice  glacier  buildings  sea       glacier   sea       glacier   forest  glacier   glacier  sea
Bruno  glacier  street     mountain  mountain  sea       mountain  street  mountain  glacier  street
Chloé  glacier  buildings  mountain  glacier   street    glacier   forest  glacier   glacier  sea
David  glacier  buildings  sea       mountain  buildings mountain  street  mountain  glacier  sea
Emma   glacier  street     sea       glacier   sea       glacier   forest  glacier   glacier  street
""")
if SEUL and etiquettes:
  annotations["Moi"] = verifie("Moi", etiquettes)
print(f"{len(annotations)} annotateurs : {', '.join(annotations)}")

## Accord inter-annotateurs

L'accord brut $p_o$ (la part des paires d'annotateurs qui ont choisi la même classe, en moyenne sur les photos) compte aussi les accords dus au hasard. Le kappa de Fleiss retire l'accord $p_e$ attendu par hasard : $\kappa = \frac{p_o - p_e}{1 - p_e}$ (frame « Accord inter-annotateurs »). Il étend à $n$ annotateurs le kappa de Cohen, qui en compare deux.

`statsmodels.stats.inter_rater.fleiss_kappa` le calcule à partir du tableau des votes : une ligne par photo, une colonne par classe, et dans chaque case le nombre d'annotateurs qui ont choisi cette classe pour cette photo. Écrivez `votes(annotations)`, qui renvoie ce tableau, et `kappa(annotations)`, qui renvoie le kappa de Fleiss, puis affichez le kappa des annotations.

Lisez-le avec l'échelle de Landis et Koch (1977), qui n'est qu'une convention : ≤ 0,20 faible, 0,21–0,40 passable, 0,41–0,60 modéré, 0,61–0,80 substantiel, > 0,80 presque parfait. Sa valeur dépend du nombre de classes et des photos choisies : il sert surtout à comparer deux mesures faites dans les mêmes conditions.

**Documentation utile**

- [`statsmodels.stats.inter_rater.aggregate_raters`](https://www.statsmodels.org/stable/generated/statsmodels.stats.inter_rater.aggregate_raters.html)
- [`statsmodels.stats.inter_rater.fleiss_kappa`](https://www.statsmodels.org/stable/generated/statsmodels.stats.inter_rater.fleiss_kappa.html)

In [ ]:
def votes(annotations):
  """Le tableau des votes : une ligne par image, une colonne par classe."""
  ...


def kappa(annotations):
  """Le kappa de Fleiss des annotations."""
  ...


# Votre code ici

### Conseils supplémentaires

`aggregate_raters` prend un tableau d'entiers, une ligne par photo et une colonne par annotateur, chaque case le numéro d'une classe (`classes.index(etiquette)`). Les annotations ont une liste par annotateur : construisez le tableau dans ce sens, puis transposez-le avec `.T`. `n_cat=len(classes)` donne une colonne à chaque classe, même quand personne ne l'a choisie. `aggregate_raters` renvoie le tableau des votes et la liste des classes : `tableau, _ = aggregate_raters(...)`. Avec les annotations d'exemple seules, le kappa vaut 0,36.

```python
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa


def votes(annotations):
  codes = numpy.array(
    [
      [classes.index(etiquette) for etiquette in etiquettes]
      for etiquettes in annotations.values()
    ]
  )
  tableau, _ = aggregate_raters(..., n_cat=...)
  return tableau


def kappa(annotations):
  return fleiss_kappa(...)


print(kappa(annotations))
```

### Solution

In [ ]:
from statsmodels.stats.inter_rater import aggregate_raters, fleiss_kappa


def votes(annotations):
  """Le tableau des votes : une ligne par image, une colonne par classe."""
  codes = numpy.array(
    [
      [classes.index(etiquette) for etiquette in etiquettes]
      for etiquettes in annotations.values()
    ]
  )
  tableau, _ = aggregate_raters(codes.T, n_cat=len(classes))
  return tableau


def kappa(annotations):
  """Le kappa de Fleiss des annotations."""
  return fleiss_kappa(votes(annotations))


print(f"Kappa de Fleiss : {kappa(annotations):.2f}")

## Accord par photo

Le kappa résume l'accord sur toutes les photos ; pour réviser le guide, il faut savoir lesquelles ont divisé. Écrivez `accord_image(annotations)`, qui renvoie, pour chaque photo, la part des paires d'annotateurs qui ont choisi la même classe : avec $n$ annotateurs et $n_{ij}$ votes pour la classe $j$ sur la photo $i$,

$$p_i = \frac{\sum_j n_{ij} (n_{ij} - 1)}{n (n - 1)}$$

La moyenne des $p_i$ est l'accord brut $p_o$ du kappa. Affichez ensuite les photos de la moins à la plus consensuelle, avec `affiche_votes(votes(annotations), accord_image(annotations))` : celles sans unanimité sont en rouge.

In [ ]:
def accord_image(annotations):
  """Pour chaque image, la part des paires d'annotateurs d'accord."""
  ...


# Votre code ici

### Conseils supplémentaires

Le tableau des votes est un tableau numpy de forme (photos, classes) : `tableau * (tableau - 1)` s'applique case par case, et `.sum(axis=1)` somme chaque ligne, donc chaque photo. $n$ est le nombre d'annotateurs, `len(annotations)`. Avec les annotations d'exemple seules, la photo 0 a un accord de 100 % et la photo 4 de 30 %.

```python
def accord_image(annotations):
  tableau = votes(annotations)
  n = len(annotations)
  return ...


affiche_votes(votes(annotations), accord_image(annotations))
```

### Solution

In [ ]:
def accord_image(annotations):
  """Pour chaque image, la part des paires d'annotateurs d'accord."""
  tableau = votes(annotations)
  n = len(annotations)
  return (tableau * (tableau - 1)).sum(axis=1) / (n * (n - 1))


affiche_votes(votes(annotations), accord_image(annotations))

## Révision du guide

Reprenez les 2 ou 3 photos les moins consensuelles. Pour chacune :

- Quelle règle du guide était ambiguë, ou manquait ?
- Quelle règle aurait permis à tout le monde de choisir la même classe ?

Écrivez-en une version 2 du guide dans la cellule de texte suivante : des définitions qui tranchent explicitement les cas limites, et un ordre de priorité quand deux classes s'appliquent à la même photo. En groupe, faites-le ensemble ; seul, révisez votre propre guide.

*Votre guide, version 2*

### Solution

Un exemple de guide révisé, celui qu'ont suivi les annotations d'exemple du deuxième tour. Avec elles, les désaccords portaient sur la glace et la neige (glacier ou montagne ?), l'eau visible à côté d'autre chose (une jetée, une falaise, une promenade : mer ?) et les routes (rue, bâtiments ou forêt ?).

Appliquez la première règle qui correspond à la photo :

1. `glacier` : une masse de glace (bleutée, fissurée, en langue ou en front) est bien visible. La neige sur des sommets ne suffit pas.
2. `sea` : l'eau (mer ou lac) occupe au moins un tiers de la photo, même avec une jetée, des falaises ou une promenade.
3. `mountain` : le relief (sommets, pentes, falaises) domine, enneigé ou non.
4. `street` : vue depuis le sol, une rue, une route ou un chemin mène dans la photo, même entre des arbres ou des bâtiments.
5. `buildings` : des bâtiments dominent, y compris une ville vue d'en haut.
6. `forest` : des arbres dominent, et aucune voie ne mène dans la photo.

## Deuxième tour d'annotation

Annotez à nouveau les 10 photos avec le guide révisé, toujours sans en discuter, puis mettez les annotations en commun comme au premier tour : en groupe, collez les lignes du groupe à la place des annotations d'exemple ; seul, remplissez `etiquettes_v2`. La cellule d'après compare les deux tours.

In [ ]:
etiquettes_v2 = []  # À compléter, avec votre guide révisé

annotations_v2 = lit_annotations("""
Alice  glacier  buildings  sea       glacier   sea  glacier   street  mountain  glacier  sea
Bruno  glacier  buildings  sea       glacier   sea  mountain  street  mountain  glacier  sea
Chloé  glacier  buildings  sea       glacier   sea  glacier   street  mountain  glacier  sea
David  glacier  buildings  mountain  glacier   sea  glacier   street  mountain  glacier  sea
Emma   glacier  buildings  sea       glacier   sea  glacier   street  mountain  glacier  street
""")
if SEUL and etiquettes_v2:
  annotations_v2["Moi"] = verifie("Moi", etiquettes_v2)

In [ ]:
print(f"Kappa de Fleiss, tour 1 : {kappa(annotations):.2f}")
print(f"Kappa de Fleiss, tour 2 : {kappa(annotations_v2):.2f}")
x = numpy.arange(len(ids))
fig, ax = plt.subplots(figsize=(10, 3.5))
ax.bar(x - 0.2, accord_image(annotations), width=0.4, label="Tour 1 (guide v1)")
ax.bar(x + 0.2, accord_image(annotations_v2), width=0.4, label="Tour 2 (guide v2)")
ax.set_xticks(x, [f"Image {i}" for i in x])
ax.set_ylim(0, 1.05)
ax.yaxis.set_major_formatter(lambda y, _: f"{y:.0%}")
ax.set_ylabel("Paires d'annotateurs d'accord")
ax.legend(loc="lower center", bbox_to_anchor=(0.5, 1), ncol=2, frameon=False)
plt.tight_layout()

Si le guide révisé a levé les ambiguïtés, le kappa augmente. Deux précautions avant de conclure :

- Les mêmes photos ont été annotées deux fois, après en avoir discuté : une partie du gain vient de la discussion elle-même. Dans un vrai projet, on vérifie le nouveau guide sur des photos que personne n'a encore vues.
- Avec 10 photos et quelques annotateurs, la mesure est très incertaine : un seul vote change nettement le kappa.

Une photo sur laquelle le désaccord persiste est peut-être réellement ambiguë (elle relève de deux classes à la fois) : c'est une information sur le problème, pas seulement sur le guide. On peut alors la faire trancher par un expert, accepter plusieurs étiquettes, ou revoir la définition des classes.

## Performance humaine de référence

La performance humaine de référence (*Human Level Performance*, HLP) se mesure en comparant chaque annotateur à la majorité des autres (sa limite : frame « HLP faussée par une convention — Solution ») : sur chaque photo, on prend la classe la plus choisie par les autres annotateurs, et on compte la part des photos où l'annotateur a choisi celle-là.

Écrivez `mesure_performance_humaine(annotations)`, qui renvoie ce score pour chaque annotateur, dans un dictionnaire nom → score, et `mesure_performance_triviale(annotations)`, le score d'un annotateur trivial, qui donne toujours la classe la plus fréquente de toutes les annotations, face à la majorité de tous les annotateurs : c'est la même règle, il ne fait pas partie du groupe et les autres, pour lui, sont donc tous les annotateurs. Affichez la HLP moyenne et le score de l'annotateur trivial, au premier tour (`annotations`) et au deuxième (`annotations_v2`).

**Documentation utile**

- [`numpy.ndarray.argmax`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.argmax.html)

In [ ]:
def mesure_performance_humaine(annotations):
  """Pour chaque annotateur, la part des images où il a choisi la classe majoritaire des autres."""
  ...


def mesure_performance_triviale(annotations):
  """La même mesure pour un annotateur qui donne toujours la classe la plus fréquente."""
  ...


# Votre code ici

### Conseils supplémentaires

Les votes des autres sont ceux de `votes(annotations)`, moins ceux de l'annotateur : avec `choix`, le tableau des numéros de ses classes, `autres[numpy.arange(len(choix)), choix] -= 1` retire un vote par photo. `autres.argmax(axis=1)` donne alors la classe majoritaire des autres sur chaque photo (en cas d'égalité, la première dans l'ordre de `classes`), et `(... == choix).mean()` la part des photos où il l'a choisie. Pour l'annotateur trivial, la classe la plus fréquente est `tableau.sum(axis=0).argmax()`, et la majorité de tous les annotateurs `tableau.argmax(axis=1)`. Avec les annotations d'exemple seules, au premier tour, Alice obtient 90 %, Bruno 30 %, et l'annotateur trivial 50 %.

```python
def mesure_performance_humaine(annotations):
  tableau = votes(annotations)
  scores = {}
  for nom, etiquettes in annotations.items():
    choix = numpy.array([classes.index(etiquette) for etiquette in etiquettes])
    autres = tableau.copy()
    autres[numpy.arange(len(choix)), choix] -= 1
    scores[nom] = ...
  return scores


def mesure_performance_triviale(annotations):
  tableau = votes(annotations)
  majorite = ...
  return ...


for annotations_tour in [annotations, annotations_v2]:
  scores = mesure_performance_humaine(annotations_tour)
  print(
    numpy.mean(list(scores.values())), mesure_performance_triviale(annotations_tour)
  )
```

### Solution

In [ ]:
def mesure_performance_humaine(annotations):
  """Pour chaque annotateur, la part des images où il a choisi la classe majoritaire des autres."""
  tableau = votes(annotations)
  scores = {}
  for nom, etiquettes in annotations.items():
    choix = numpy.array([classes.index(etiquette) for etiquette in etiquettes])
    autres = tableau.copy()
    autres[numpy.arange(len(choix)), choix] -= 1
    scores[nom] = (autres.argmax(axis=1) == choix).mean()
  return scores


def mesure_performance_triviale(annotations):
  """La même mesure pour un annotateur qui donne toujours la classe la plus fréquente."""
  tableau = votes(annotations)
  majorite = tableau.sum(axis=0).argmax()
  return (tableau.argmax(axis=1) == majorite).mean()


for nom_tour, annotations_tour in [
  ("Tour 1", annotations),
  ("Tour 2", annotations_v2),
]:
  scores = mesure_performance_humaine(annotations_tour)
  print(
    f"{nom_tour} : HLP moyenne {numpy.mean(list(scores.values())):.0%}, "
    f"annotateur trivial {mesure_performance_triviale(annotations_tour):.0%}"
  )
  print("  " + ", ".join(f"{nom} {score:.0%}" for nom, score in scores.items()))

## Impact sur le niveau de performance humain

Comparez la HLP à l'annotateur trivial, au premier puis au deuxième tour. Quel est l'impact d'une mauvaise annotation sur la HLP ? Un algorithme d'apprentissage subit-il les mauvaises annotations de la même manière ?

*Votre réponse ici*

### Solution

Quelques éléments de réponse :

- Avec les annotations d'exemple, la HLP moyenne du premier tour (62 %) dépasse à peine l'annotateur trivial (50 %, qui répond toujours `glacier`) : un modèle qui reproduirait la classe majoritaire « atteindrait » presque le niveau humain sans rien comprendre. Au deuxième tour, la HLP monte (94 %) et l'annotateur trivial reste loin derrière (40 %).
- La HLP se mesure en comparant les annotations d'un humain à une référence, ici la majorité des autres annotateurs. Avec un guide ambigu, des annotateurs compétents se contredisent : la HLP mesurée baisse, mais elle reflète alors l'incohérence des étiquettes plus que la difficulté réelle de la tâche. Elle estime donc mal l'erreur de Bayes.
- Un modèle peut alors sembler « dépasser le niveau humain » simplement parce qu'il reproduit mieux l'étiquette majoritaire. Ce n'est pas un vrai progrès : clarifier le guide fait remonter la HLP et révèle la marge qui reste.
- Un algorithme d'apprentissage ne subit pas toutes les erreurs de la même manière :
  - des erreurs aléatoires et peu nombreuses se compensent en partie sur beaucoup d'exemples, mais elles demandent plus de données et abaissent les performances atteignables ;
  - des erreurs systématiques (par exemple, toujours appeler `glacier` une montagne enneigée) sont apprises telles quelles : le modèle reproduit fidèlement la règle implicite des annotateurs ;
  - des étiquettes bruitées dans le jeu de test rendent l'évaluation elle-même peu fiable.
- Améliorer la cohérence de l'annotation (guide, double annotation des cas ambigus) est souvent plus rentable que de changer de modèle.